# Colab T4 Server — Qwen2.5-Coder-1.5B + LoRA Adapter (FastAPI + ngrok)
Run these 3 cells on Google Colab (Runtime -> Change runtime type -> **T4 GPU**):
1. **Cell 1:** Mount Google Drive & merge LoRA adapter into float16 model (fast ~1.5s inference, no 4-bit hang)
2. **Cell 2:** Start FastAPI `/health` and `/generate` server with thread lock
3. **Cell 3:** Start ngrok tunnel and print your `.env` block to copy-paste into VS Code

In [ ]:
# CELL 1: Install packages, mount Drive, load & merge LoRA adapter in float16
!pip install -q fastapi uvicorn pyngrok transformers peft accelerate

import os, time, threading, re, torch
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

drive.mount('/content/drive')

BASE_MODEL   = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/sql_lora_adapter_5k"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map={"": 0})

if os.path.isdir(ADAPTER_PATH):
    print("Loading fine-tuned LoRA adapter from:", ADAPTER_PATH)
    model = PeftModel.from_pretrained(base, ADAPTER_PATH, local_files_only=True)
    model = model.merge_and_unload()  # merge into fp16 so inference takes ~1.5s and never hangs!
else:
    print("WARNING: Adapter folder not found at", ADAPTER_PATH, "— using base model")
    model = base

model.eval()
GEN_LOCK = threading.Lock()

SYSTEM_PROMPT = (
    "Convert questions into valid SQLite SQL "
    "using only the given schema. Return only SQL."
)

def generate_sql(schema_text: str, question: str, max_tokens: int = 200) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Schema:\n{schema_text}\nQuestion:\n{question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    with GEN_LOCK, torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=min(int(max_tokens), 256),
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            max_time=45,
        )
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
    m = re.search(r"```(?:sql)?\s*(.*?)```", raw, re.S | re.I)
    sql = (m.group(1) if m else raw.split("```")[0]).strip()
    print(f"[MODEL] Generated in {time.time() - t0:.2f}s: {sql}", flush=True)
    return sql

print("Model ready! Test output:", generate_sql("students(student_id, name, year)", "Count all students"))

In [ ]:
# CELL 2: Start FastAPI Server on port 8000
import secrets, threading, time, uvicorn
from fastapi import FastAPI, HTTPException, Header

# Set a fixed 32-char token (or enter your own >=24 char token)
API_TOKEN = "a0Bc8jMGzWKq-Oz6vZ2-PNsrcKbHBIHIEZyfwfcKFCk"

app = FastAPI(title="Text-to-SQL API", version="1.0")

@app.get("/health")
def health():
    return {"status": "running", "model": BASE_MODEL, "adapter": "merged-fp16"}

@app.post("/generate")
def generate(payload: dict, authorization: str = Header(default=None)):
    expected = f"Bearer {API_TOKEN}"
    if not authorization or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail="Invalid or missing API token")

    schema_text = payload.get("schema", "")
    question    = payload.get("question", "")
    max_tokens  = payload.get("max_tokens", 200)

    if not isinstance(schema_text, str) or not schema_text.strip():
        raise HTTPException(status_code=400, detail="Schema is required")
    if not isinstance(question, str) or not question.strip():
        raise HTTPException(status_code=400, detail="Question is required")

    try:
        sql = generate_sql(schema_text, question, max_tokens=max_tokens)
        return {"success": True, "sql": sql}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print("FastAPI server started on port 8000!")

In [ ]:
# CELL 3: Start ngrok tunnel and print exact .env values for VS Code
import getpass
from pyngrok import ngrok

NGROK_TOKEN = getpass.getpass("Paste your ngrok Authtoken (from dashboard.ngrok.com): ").strip()
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()

tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url

print("\n" + "=" * 60)
print("COPY AND PASTE THIS EXACT BLOCK INTO YOUR LOCAL .env FILE:")
print("=" * 60)
print(f"LLM_BACKEND=colab")
print(f"COLAB_API_URL={PUBLIC_URL}")
print(f"COLAB_API_TOKEN={API_TOKEN}")
print("=" * 60)